In [12]:
# from google.colab import drive
import torch
import sys
import numpy as np
import math
import time
import torch.nn.functional as F
from transformers import get_cosine_schedule_with_warmup
# drive.mount('/content/gdrive', force_remount=True)


#base_dir = "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/Final Project"
# base_dir = "/content/gdrive/MyDrive/Final Project"
base_dir = ""
# sys.path.append(base_dir)

device = torch.device("cuda" if torch.cuda.is_available()
                else ("mps" if torch.backends.mps.is_available()
                else "cpu"))
#device = torch.device("meta")
print(F"Device set to {device}")


Device set to mps


In [13]:
import importlib
import Models.GPT_Model as GPT_Model
import Datasets.DataLoader as DataLoader_Lib

importlib.reload(GPT_Model)
importlib.reload(DataLoader_Lib)

from Models.GPT_Model import GPT2_Lag, GPTConfig
from Models.BERT_Model import BERT_Lag, BERTConfig
from Datasets.DataLoader import TinyShakespeareDataLoader, TinyStoriesDataLoader, CombinedBinDataLoader

In [14]:
# importlib.reload(GPT_Model)
# B = 4 
# block_size = 128

# loader = TinyStoriesDataLoader(max_length= block_size, batch_size=B)
# train_loader, val_loader = loader.get_data()


In [ ]:
batch_per_iter = 8 # Adjust batch size based on your Colab GPU memory
grad_acc_factor = 16
eff_batch_size = batch_per_iter * grad_acc_factor
block_size = 256
warmup_steps = 200
num_steps_train = 3000
num_steps_val = 10
weight_decay = .1
dropout = 0.1
tokens_per_step = eff_batch_size * block_size

config = BERTConfig(num_heads = 12,
  num_layers = 12,
  vocab_size = 50257,
  embedding_dim = 768,
  block_size = block_size,
  dropout = dropout,
  weight_decay = weight_decay,
  pad_token_id=50256)

model = BERT_Lag(config, device)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=weight_decay)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=num_steps_train
)

# Config = GPTConfig(num_heads = 12,
#   num_layers = 12,
#   vocab_size = 50257,
#   embedding_dim = 768,
#   block_size = block_size,
#   lag_behind = 1,
#   dropout = .1,
#   pad_token_id=loader.pad_token())

# model = GPT2_Lag(Config, device)
# model.to(device)
# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

TypeError: BFloat16 is not supported on MPS

In [ ]:
train_loader, val_loader = CombinedBinDataLoader.create_loaders(
    'Datasets/combined_dataset.bin', batch_per_iter, block_size, config, seed=0
)

Initialized loader with 6,696 chunks of size 4097.
Initialized loader with 745 chunks of size 4097.


In [ ]:
@torch.no_grad()
def estimate_loss(model, loader, device, eval_iters=10):
    model.eval()

    losses = []

    for _ in range(eval_iters):
        x, y, _ = loader.get_data()
        x, y = x.to(device), y.to(device)

        loss_fwd, loss_bwd = model(x, y)
        losses.append(loss_fwd.item())
  
    model.train()

    avg = sum(losses) / len(losses)
    ppl = torch.exp(torch.tensor(avg)).item()

    return loss_fwd, ppl, loss_bwd



def train_loop(model, optimizer, scheduler, device, train_loader, val_loader,
               num_steps_train, num_steps_val):

    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    loss_fwd, ppl, loss_bwd = estimate_loss(model, val_loader, device, num_steps_val)
    loss_bwd = torch.tensor(-100)
    print(f"Step    0 | Val FWD: {loss_fwd:.4f} | PPL: {ppl:.2f} | Val BWD: {loss_bwd:.4f}")
    start = time.time()
    tokens_seen = 0

    #Define some useful constants

    for step in range(num_steps_train):
        if step % 50 == 0 and step > 0:
            loss_fwd, ppl, loss_bwd = estimate_loss(model, val_loader, device, num_steps_val)
            loss_bwd = torch.tensor(-100)
            print(f"Step {step:4d} | Val FWD: {loss_fwd:.4f} | PPL: {ppl:.2f} | Val BWD: {loss_bwd:.4f}")
            if device == torch.device('mps'):
                torch.mps.empty_cache()

        optimizer.zero_grad()
        for _ in range(grad_acc_factor):
            x, y, _ = train_loader.get_data()
            x, y = x.to(device), y.to(device)

            loss_fwd, loss_bwd = model(x, y)
            loss_bwd = torch.tensor(-100)
            loss = (loss_fwd + 0 * loss_bwd) / grad_acc_factor
            loss.backward()
        optimizer.step()
        scheduler.step()
        tokens_seen += tokens_per_step
        if step % 10 == 0:   
            stop = time.time()
            print(f"step {step:4d} | tokens {tokens_seen:,} | Train FWD loss {loss_fwd.item():.4f} | Train BWD loss {loss_bwd.item():.4f} | Time Since Last Train Print {(stop-start):.4f} seconds")
            start = time.time()


In [ ]:
steps_in_train_loader = (len(train_loader.split_starts) * train_loader.chunk_size)//tokens_per_step

train_loop(model, optimizer, scheduler, device, train_loader, val_loader, steps_in_train_loader, grad_acc_factor)

Trainable parameters: 123,849,984
Step    0 | Val FWD: 10.9972 | PPL: 59704.57 | Val BWD: -100.0000


RuntimeError: User specified an unsupported autocast device_type 'mps'